# Part 2 - MCTNet avec CSV temporel reel

Pipeline compatible avec `ARKANSAS_FULL_TEMPORAL.csv` et `CALIFORNIA_FULL_TEMPORAL.csv` :
- labels integres au CSV
- Sentinel-2 temporel en `t0...t35`
- climat temporel
- covariables statiques separees puis broadcastees dans le modele


In [ ]:
import subprocess
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    'numpy',
    'pandas',
    'matplotlib',
    'torch',
])

import json
from types import SimpleNamespace

import numpy as np
import pandas as pd


In [ ]:
PROJECT_DIR = Path('/content/drive/MyDrive/New project/python')
SEARCH_ROOTS = [
    Path('/content/drive/MyDrive'),
    Path('/content/drive/MyDrive/projoRN'),
    Path('/content/drive/MyDrive/New project'),
]
PROCESSED_ENV_DIR = PROJECT_DIR / 'processed_env_real_csv'
EDA_DIR = PROJECT_DIR / 'eda_real_csv'
ABLATION_DIR = PROJECT_DIR / 'ablation_real_csv'

for folder in [PROCESSED_ENV_DIR, EDA_DIR, ABLATION_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))


def resolve_existing_path(candidate_names, search_roots):
    for root in search_roots:
        for candidate_name in candidate_names:
            candidate = root / candidate_name
            if candidate.exists():
                return candidate

    for root in search_roots:
        for candidate_name in candidate_names:
            matches = list(root.rglob(candidate_name))
            if matches:
                return matches[0]
    return None


DATASETS = {
    'arkansas': {
        'data_csv': resolve_existing_path(
            ['ARKANSAS_FULL_TEMPORAL.csv', 'mctnet_AR_2021.csv'],
            SEARCH_ROOTS,
        ),
        'labels_csv': None,
    },
    'california': {
        'data_csv': resolve_existing_path(
            ['CALIFORNIA_FULL_TEMPORAL.csv', 'mctnet_CA_2021.csv'],
            SEARCH_ROOTS,
        ),
        'labels_csv': None,
    },
}

for state_slug, paths in DATASETS.items():
    print(f'\nState: {state_slug}')
    print('  data_csv  :', paths['data_csv'])
    print('  labels_csv:', paths['labels_csv'])


In [ ]:
missing_data = [state for state, paths in DATASETS.items() if paths['data_csv'] is None]
if missing_data:
    raise FileNotFoundError('CSV de donnees manquants pour: ' + ', '.join(missing_data))


In [ ]:
from build_mctnet_env_dataset import build_env_dataset_bundle, save_bundle

for state_slug, paths in DATASETS.items():
    bundle, metadata = build_env_dataset_bundle(
        csv_path=paths['data_csv'],
        labels_csv_path=paths['labels_csv'],
        normalize_reflectance=True,
        reflectance_scale=10000.0,
        normalize_environment=True,
        split_seed=2021,
    )

    output_npz = PROCESSED_ENV_DIR / f'{metadata["state_slug"]}_mctnet_env_dataset.npz'
    output_json = PROCESSED_ENV_DIR / f'{metadata["state_slug"]}_mctnet_env_dataset.json'
    save_bundle(bundle, metadata, output_npz, output_json)

    print(f'[{metadata["state_slug"]}] saved')
    print(f'  x_train shape: {bundle["x_train"].shape}')
    print(f'  dynamic_env_train shape: {bundle["dynamic_env_train"].shape}')
    print(f'  static_env_train shape: {bundle["static_env_train"].shape}')
    print(f'  valid_mask_train shape: {bundle["valid_mask_train"].shape}')
    print(f'  classes: {metadata["class_name_to_index"]}')


In [ ]:
for state_slug in DATASETS:
    npz_path = PROCESSED_ENV_DIR / f'{state_slug}_mctnet_env_dataset.npz'
    json_path = PROCESSED_ENV_DIR / f'{state_slug}_mctnet_env_dataset.json'
    if not npz_path.exists() or not json_path.exists():
        continue

    with np.load(npz_path, allow_pickle=True) as data:
        bundle = {key: data[key] for key in data.files}
    metadata = json.loads(json_path.read_text(encoding='utf-8'))

    print(f'\nState: {state_slug}')
    print('x_train:', bundle['x_train'].shape)
    print('dynamic_env_train:', bundle['dynamic_env_train'].shape)
    print('static_env_train:', bundle['static_env_train'].shape)
    print('valid_mask_train:', bundle['valid_mask_train'].shape)
    print('y_train:', bundle['y_train'].shape)
    print('nb_classes:', metadata['num_classes'])
    print('sequence_length:', metadata['sequence_length'])
    print('spectral_channels:', metadata['feature_shape_per_sample'][-1])
    print('all-input dim:', metadata['ablation_configs']['all']['input_dim'])
    print('ablation configs:', json.dumps(metadata['ablation_configs'], indent=2))


In [ ]:
from environmental_covariate_eda import run_eda

eda_artifacts = {}
for state_slug, paths in DATASETS.items():
    state_eda_dir = EDA_DIR / state_slug
    artifacts = run_eda(
        csv_path=paths['data_csv'],
        output_dir=state_eda_dir,
        labels_csv_path=paths['labels_csv'],
    )
    eda_artifacts[state_slug] = artifacts
    print(f'\nEDA artifacts for {state_slug}:')
    print(json.dumps(artifacts, indent=2))


In [ ]:
from IPython.display import Image, display

for state_slug, artifacts in eda_artifacts.items():
    print(f'\nEDA preview: {state_slug}')
    for key in ['pearson_heatmap_png', 'class_means_heatmap_png', 'eta_barplot_png']:
        image_path = Path(artifacts[key])
        if image_path.exists():
            print(key)
            display(Image(filename=str(image_path)))


In [ ]:
from run_ablation_study import run_ablation_experiment

CFG = SimpleNamespace(
    epochs=200,
    batch_size=32,
    learning_rate=1e-3,
    weight_decay=1e-4,
    dropout=0.1,
    n_stages=3,
    n_heads=5,
    kernel_size=3,
    seed=2021,
    num_workers=0,
    early_stopping_patience=25,
    cpu=False,
    no_alpe=False,
    no_mask=False,
    no_cnn=False,
    no_trans=False,
    boost_classes=['rice', 'cotton'],
    boost_factor=1.5,
    states=['arkansas', 'california'],
    configs=['climate', 'soil', 'topography', 'all'],
)

ablation_results = run_ablation_experiment(
    processed_env_dir=PROCESSED_ENV_DIR,
    output_dir=ABLATION_DIR,
    states=CFG.states,
    args=CFG,
)

print(json.dumps(ablation_results, indent=2))

summary_rows = []
for state_name, config_metrics in ablation_results.items():
    for config_name, metrics in config_metrics.items():
        summary_rows.append({
            'state': state_name,
            'config': config_name,
            'best_epoch': int(metrics.get('best_epoch', 0)),
            'best_test_oa': round(metrics.get('oa', 0.0), 4),
            'best_test_kappa': round(metrics.get('kappa', 0.0), 4),
            'best_test_f1': round(metrics.get('macro_f1', 0.0), 4),
            'best_val_oa': round(metrics.get('best_val_oa', 0.0), 4),
            'best_val_kappa': round(metrics.get('best_val_kappa', 0.0), 4),
            'best_val_f1': round(metrics.get('best_val_macro_f1', 0.0), 4),
            'last_epoch': int(metrics.get('last_epoch', 0)),
            'last_val_oa': round(metrics.get('last_val_oa', 0.0), 4),
            'last_val_kappa': round(metrics.get('last_val_kappa', 0.0), 4),
            'last_val_f1': round(metrics.get('last_val_macro_f1', 0.0), 4),
            'last_test_oa': round(metrics.get('last_test_oa', 0.0), 4),
            'last_test_kappa': round(metrics.get('last_test_kappa', 0.0), 4),
            'last_test_f1': round(metrics.get('last_test_macro_f1', 0.0), 4),
        })

results_df = pd.DataFrame(summary_rows)
comparison_columns = [
    'state',
    'config',
    'best_epoch',
    'best_test_oa',
    'best_test_kappa',
    'best_test_f1',
    'best_val_oa',
    'best_val_kappa',
    'best_val_f1',
]

if not results_df.empty:
    results_df = results_df.sort_values(['state', 'config']).reset_index(drop=True)
    print('Tableau complet des resultats')
    display(results_df)

    print('Classement par OA test')
    oa_rank_df = results_df[comparison_columns].sort_values(
        ['state', 'best_test_oa', 'best_test_kappa', 'best_test_f1'],
        ascending=[True, False, False, False],
    ).reset_index(drop=True)
    display(oa_rank_df)

    print('Classement par Kappa test')
    kappa_rank_df = results_df[comparison_columns].sort_values(
        ['state', 'best_test_kappa', 'best_test_oa', 'best_test_f1'],
        ascending=[True, False, False, False],
    ).reset_index(drop=True)
    display(kappa_rank_df)

    best_by_state_oa = oa_rank_df.groupby('state', as_index=False).first()
    print('Meilleure configuration par etat selon OA test')
    display(best_by_state_oa)

    best_by_state_kappa = kappa_rank_df.groupby('state', as_index=False).first()
    print('Meilleure configuration par etat selon Kappa test')
    display(best_by_state_kappa)

summary_csv = ABLATION_DIR / 'ablation_summary.csv'
if summary_csv.exists():
    df = pd.read_csv(summary_csv)
    print('Resume CSV sauvegarde')
    display(df.sort_values(['state', 'config']).reset_index(drop=True))

for metric in ['oa', 'macro_f1', 'kappa']:
    image_path = ABLATION_DIR / f'ablation_{metric}_barplot.png'
    if image_path.exists():
        display(Image(filename=str(image_path)))

for state_slug in CFG.states:
    state_dir = ABLATION_DIR / state_slug
    for image_path in sorted(state_dir.glob('confusion_matrix_*.png')):
        print(image_path.name)
        display(Image(filename=str(image_path)))
